# Custom tokenizer


In [1]:
import regex as re
from tokenizers import Tokenizer, AddedToken, pre_tokenizers, NormalizedString, PreTokenizedString
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from tokenizers.pre_tokenizers import PreTokenizer, Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

In [2]:
# träningsfiler
files = ["svensk_text_1.txt",
        "svensk_text_2.txt",
        "svensk_text_3.txt",
        ]

# variables
vocab_size = 10000 
min_frequency = 4

# Define special tokens
special_tokens = [
    "[PAD]",   # Padding token
    "[UNK]",   # Unknown token
    "[CLS]",   # Classification token
    "[SEP]",   # Separator token
    "[MASK]",  # Masking token
]

# Initilize trainer
trainer = WordPieceTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
    continuing_subword_prefix="##",
)

In [3]:
text = "Det här är en exempelmening på svenska med åäö och sammansatta ord som e-post."
text = "Elmontörer kommer att spela en viktig roll i framtidens samhälle."
text = "Elmaterial har en viktig roll i framtidens samhälle att spela."
# text = "Byggmontör kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggarbete kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggmaterial kommer att spela en viktig roll i framtidens samhälle."
text = "Han behöver elmaterial för sitt elarbete som elmontör."
text

'Han behöver elmaterial för sitt elarbete som elmontör.'

In [4]:
# Build a tokenizer
bert0_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert0_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bert0_tokenizer.pre_tokenizer = Whitespace()

# Train the tokenizer on the provided files
bert0_tokenizer.train(files, trainer)

In [5]:
# Build a tokenizer
bert1_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert1_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bert1_tokenizer.pre_tokenizer = Whitespace()

# Add "el" as a special token (important for recognition)
bert1_tokenizer.add_tokens(["el"])
bert1_tokenizer.add_tokens([AddedToken("el", single_word=False, normalized=True, special=False)])
# bert1_tokenizer.add_tokens([AddedToken("el", single_word=True, normalized=True, special=False)])

# Train the tokenizer on the provided files
bert1_tokenizer.train(files, trainer)

In [6]:
class ElPrefixPreTokenizer:
    def pre_tokenize(self, pretok: PreTokenizedString):
        # Let's call split on the PreTokenizedString to split
        pretok.split(self.custom_split)
        # Here we can call `pretok.split` multiple times if we want to apply
        # different algorithm, but we generally just need to call it once.
        # pretok.split(self.another_split_function)
    
    def custom_split(self, i: int, normalized_string: NormalizedString) -> list[NormalizedString]:
        text = str(normalized_string)
        splits = []
        if text.lower().startswith("el") and len(text) > 2:  # Check for 'el' prefix (case-insensitive) and minimum word length
            splits.append(NormalizedString("el"))
            splits.append(NormalizedString(text[2:]))  # Rest of the word
        else:
            splits.append(normalized_string)  # No 'el' prefix, keep as is
        return splits

In [7]:
# Initialize tokenizer
bert2_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert2_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bert2_tokenizer.pre_tokenizer = PreTokenizer.custom(ElPrefixPreTokenizer())

# Train the tokenizer on the provided files
bert2_tokenizer.train(files, trainer)

In [26]:
# Custom regex-based pre-tokenizer function
pattern = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"
compiled_pattern = re.compile(pattern)

class CustomPreTokenizer:
    def pre_tokenize(self, pretok: PreTokenizedString):
        pretok.split(self.custom_regex_pretok)
    
    def custom_regex_pretok(self, i: int, normalized_string: NormalizedString) -> list[NormalizedString]:
        text = str(normalized_string)
        splits = []
        for match in compiled_pattern.finditer(text):
            token = match.group()
            start = match.start()
            splits.append(token, start)
        return splits

In [30]:
# Initialize tokenizer
bert3_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert3_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
# bert3_tokenizer.pre_tokenizer = PreTokenizer.custom(CustomPreTokenizer())

# Train the tokenizer on the provided files
bert3_tokenizer.train(files, trainer)

## Load tokenizer

In [31]:
# Test encoding
bert0_encode = bert0_tokenizer.encode(text)
bert1_encode = bert1_tokenizer.encode(text)
bert2_encode = bert2_tokenizer.encode(text)

# token information
print("Tokens for:")
print("BERT 0 - default:", bert0_encode.tokens)
print("BERT 1 - added token:", bert1_encode.tokens)
print("BERT 2 - custom split pretokenizer:", bert2_encode.tokens)

# decoded text
print("\nOriginaltext......................:", text)
print("Decoded text for:")
print("BERT 0 - default..................:", bert0_tokenizer.decode(bert0_encode.ids))
print("BERT 1 - added token..............:", bert1_tokenizer.decode(bert1_encode.ids))
print("BERT 2 - custom split pretokenizer:", bert2_tokenizer.decode(bert2_encode.ids))

Tokens for:
BERT 0 - default: ['ha', '##n', 'behöver', 'elmaterial', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
BERT 1 - added token: ['ha', '##n', 'behöver', 'el', 'material', 'för', 'sitt', 'el', 'arbete', 'som', 'el', 'montör', '.']
BERT 2 - custom split pretokenizer: ['h', '##an ', '##behöver ', '##elmaterial ', '##för ', '##sitt ', '##elarbete', '## som ', '##elmontör', '##.']

Originaltext......................: Han behöver elmaterial för sitt elarbete som elmontör.
Decoded text for:
BERT 0 - default..................: ha ##n behöver elmaterial för sitt elarbete som elmontör .
BERT 1 - added token..............: ha ##n behöver material för sitt arbete som montör .
BERT 2 - custom split pretokenizer: h ##an  ##behöver  ##elmaterial  ##för  ##sitt  ##elarbete ## som  ##elmontör ##.


In [32]:
bert1_tokenizer.get_added_tokens_decoder()

{0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}